In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# (1) 定义超参数（校长定规矩）
batch_size = 64      # 每次给 AI 发 64 张卷子
learning_rate = 0.01 # 教练纠错的力度
epochs = 3           # 题库完整刷 3 遍（为了让你快点看到结果，我们先设小一点）

# (2) 构建 transforms（统一排版规则）
# ToTensor() 把图片变成张量，Normalize() 把数据标准化，让 AI 更好消化
my_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # 这是 MNIST 的标准魔法参数，不用背
])

# (3) 下载并加载数据集（买真题库 + 上传送带）
print("正在去网上买题库，请稍等片刻...")
# 买【训练集】（平时作业）
train_dataset = datasets.MNIST(root='./data', train=True, transform=my_transform, download=True)
# 买【测试集】（期末考卷）
test_dataset = datasets.MNIST(root='./data', train=False, transform=my_transform, download=True)

# 放到传送带上（DataLoader），每次吐出 64 张，打乱顺序
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

print("题库准备完毕！传送带已启动！")


In [ ]:
# (4) 构建网络模型（物色一个好苗子）
class MyNet(nn.Module):
    def __init__(self):
        super(MyNet, self).__init__()
        # 组装零件：图片是 28x28 像素，压扁后是 784 个点。
        # 第一层：把 784 个点，映射到 128 个神经元
        self.fc1 = nn.Linear(28 * 28, 128) 
        # 激活函数：给直线加点“弯折”，让它能学复杂的特征
        self.relu = nn.ReLU() 
        # 第二层：把 128 个神经元，浓缩成 10 个选项（对应数字 0 到 9）
        self.fc2 = nn.Linear(128, 10) 

    def forward(self, x):
        # 怎么做题？
        x = x.view(-1, 28 * 28) # 第一步：把方块图片压扁成一条长线
        x = self.fc1(x)         # 第二步：经过第一层脑细胞
        x = self.relu(x)        # 第三步：经过非线性滤镜
        x = self.fc2(x)         # 第四步：得出 10 个选项的分数
        return x

# 实例化这个学生
model = MyNet()

# 定义红笔（损失函数）：CrossEntropyLoss 是专门用来做“多选题”的！
criterion = nn.CrossEntropyLoss()

# 定义教练（优化器）：还是咱们熟悉的 SGD 教练
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

print("AI 学生已就位，教练和红笔已准备好！")

In [ ]:
# (7) 开始训练和测试大循环
for epoch in range(epochs): # 题库刷 3 遍
    
    # ======= 【平时的魔鬼训练】 =======
    model.train() # 告诉学生：现在是平时练习，可以改错！
    for batch_idx, (images, labels) in enumerate(train_loader): # 从传送带上拿卷子
        
        # 1. 学生做题
        outputs = model(images) 
        # 2. 老师判卷
        loss = criterion(outputs, labels) 
        
        # 3. 练功三板斧（核心！）
        optimizer.zero_grad() # 擦黑板
        loss.backward()       # 找病根
        optimizer.step()      # 改正错误
        
        # 每做 200 批题目，打印一次进度
        if batch_idx % 200 == 0:
            print(f"第 {epoch+1} 遍刷题，进度: {batch_idx}/{len(train_loader)}，当前的错题离谱度(Loss): {loss.item():.4f}")

            
    # ======= 【周末的期末考试】 =======
    model.eval() # 告诉学生：现在是考试！不许改错（不许更新参数）！
    correct = 0  # 记录猜对了多少题
    total = 0    # 记录总共有多少题
    
    with torch.no_grad(): # 考试时收走红笔，不需要算梯度（极度重要！）
        for images, labels in test_loader:
            outputs = model(images)
            # outputs 里有 10 个分数，选分数最高的那个当做最终答案
            _, predicted = torch.max(outputs.data, 1) 
            
            total += labels.size(0) # 题目总数增加
            correct += (predicted == labels).sum().item() # 如果猜对了，答对题数加1

    accuracy = 100 * correct / total # 计算准确率
    print(f"🎉 第 {epoch+1} 遍学完后的期末考试准确率: {accuracy:.2f}% \n")

print("全部训练结束！你太牛了！")